In [2]:
! pip install optuna
! pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 20.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.8/797.8 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/6

# BigMartSales

Imports and Global Configs

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor

import optuna
import mlflow
import mlflow.sklearn

RANDOM_STATE = 42
N_SPLITS = 5

mlflow.set_experiment("BigMartSales_Prediction")


<Experiment: artifact_location='/content/mlruns/2', creation_time=1770874129715, experiment_id='2', last_update_time=1770874129715, lifecycle_stage='active', name='BigMartSales_Prediction', tags={}>

Data Loading

In [4]:
train = pd.read_csv("/content/train_v9rqX0R.csv")
test = pd.read_csv("/content/test_AbJTz2l.csv")

print(train.shape, test.shape)

(8523, 12) (5681, 11)


Feature Engineering Pipeline

In [5]:
def feature_engineering(train, test):

    train = train.copy()
    test = test.copy()

    # Handling Missing Item Weight
    item_weight_median = train.groupby("Item_Type")["Item_Weight"].median()

    for df in [train, test]:
        df["Item_Weight"] = df.apply(
            lambda x: item_weight_median[x["Item_Type"]]
            if pd.isnull(x["Item_Weight"]) else x["Item_Weight"],
            axis=1
        )

    # Visibility Handling

    train.loc[train["Item_Visibility"] == 0, "Item_Visibility"] = np.nan
    test.loc[test["Item_Visibility"] == 0, "Item_Visibility"] = np.nan

    visibility_median = train.groupby("Item_Type")["Item_Visibility"].median()

    for df in [train, test]:
        df["Item_Visibility"] = df.apply(
            lambda x: visibility_median[x["Item_Type"]]
            if pd.isnull(x["Item_Visibility"]) else x["Item_Visibility"],
            axis=1
        )

    # Fat Content Normalize + Binary Encoding

    mapping = {
        "lf": "Low Fat",
        "low fat": "Low Fat",
        "reg": "Regular",
        "regular": "Regular"
    }

    for df in [train, test]:
        df["Item_Fat_Content"] = (
            df["Item_Fat_Content"]
            .astype(str)
            .str.strip()
            .str.lower()
            .replace(mapping)
        )

    fat_encoding = {"Low Fat": 1, "Regular": 0}

    for df in [train, test]:
        df["Item_Fat_Content"] = df["Item_Fat_Content"].replace(fat_encoding)

    # Outlet Age Generation

    CURRENT_YEAR = 2013
    for df in [train, test]:
        df["Outlet_Age"] = CURRENT_YEAR - df["Outlet_Establishment_Year"]

    train.drop(columns=["Outlet_Establishment_Year"], inplace=True)
    test.drop(columns=["Outlet_Establishment_Year"], inplace=True)


    # Visibility Ratio

    visibility_mean = train.groupby("Item_Type")["Item_Visibility"].transform("mean")
    train["Visibility_Ratio"] = train["Item_Visibility"] / visibility_mean

    visibility_map = train.groupby("Item_Type")["Item_Visibility"].mean()
    test["Visibility_Ratio"] = test["Item_Visibility"] / test["Item_Type"].map(visibility_map)


    # MRP Binning
    bins = [0, 70, 140, 210, 300]

    train["MRP_Bin"] = pd.cut(train["Item_MRP"], bins=bins, labels=False)
    test["MRP_Bin"] = pd.cut(test["Item_MRP"], bins=bins, labels=False)

    # Interaction Feature

    train["Item_Outlet_Type"] = train["Item_Type"] + "_" + train["Outlet_Type"]
    test["Item_Outlet_Type"] = test["Item_Type"] + "_" + test["Outlet_Type"]

    return train, test


Prepare Modeling Data

In [9]:
from sklearn.model_selection import KFold

train_fe, test_fe = feature_engineering(train, test)

y = train_fe["Item_Outlet_Sales"]
X_raw = train_fe.drop("Item_Outlet_Sales", axis=1)
test_raw = test_fe.copy()

test_ids = test_raw[["Item_Identifier", "Outlet_Identifier"]]

X_raw.drop(columns=["Item_Identifier", "Outlet_Identifier"], inplace=True)
test_raw.drop(columns=["Item_Identifier", "Outlet_Identifier"], inplace=True)

X_encoded = pd.get_dummies(X_raw, drop_first=True)
test_encoded = pd.get_dummies(test_raw, drop_first=True)

X_encoded, test_encoded = X_encoded.align(
    test_encoded,
    join="left",
    axis=1,
    fill_value=0
)

# groups = train_fe["Outlet_Identifier"]
# gkf = GroupKFold(n_splits=N_SPLITS)



kf = KFold(n_splits=5, shuffle=True, random_state=42)

Model Evaluation Utility

In [10]:
def evaluate_model(model, X, y):
    scores = []

    for train_idx, val_idx in kf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, preds))
        scores.append(rmse)

    return np.mean(scores)


Baseline MLflow Run

In [11]:
with mlflow.start_run(run_name="GB_baseline"):

    model = GradientBoostingRegressor(
        n_estimators=72,
        learning_rate=0.05,
        max_depth=4,
        min_samples_leaf=10,
        subsample=0.8,
        random_state=RANDOM_STATE
    )

    cv_rmse = evaluate_model(model, X_encoded, y)

    mlflow.log_params(model.get_params())
    mlflow.log_metric("cv_rmse", cv_rmse)

    print("Baseline CV RMSE:", cv_rmse)


Baseline CV RMSE: 1083.928660399784


Optuna + MLflow Integration

In [12]:
def objective(trial):

    with mlflow.start_run(nested=True):

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
            "max_depth": trial.suggest_int("max_depth", 2, 6),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "random_state": RANDOM_STATE
        }

        model = GradientBoostingRegressor(**params)

        cv_rmse = evaluate_model(model, X_encoded, y)

        mlflow.log_params(params)
        mlflow.log_metric("cv_rmse", cv_rmse)

        return cv_rmse


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("Best RMSE:", study.best_value)
print("Best Params:", study.best_params)


[I 2026-02-12 05:50:35,427] A new study created in memory with name: no-name-3fdc2896-2355-497d-946b-abcd6fa99dd6
[I 2026-02-12 05:50:52,465] Trial 0 finished with value: 1086.156216293503 and parameters: {'n_estimators': 89, 'learning_rate': 0.04892276467269247, 'max_depth': 5, 'min_samples_leaf': 11, 'subsample': 0.9398805318895881}. Best is trial 0 with value: 1086.156216293503.
[I 2026-02-12 05:51:04,914] Trial 1 finished with value: 1102.025055088347 and parameters: {'n_estimators': 155, 'learning_rate': 0.1723577746676003, 'max_depth': 2, 'min_samples_leaf': 7, 'subsample': 0.8988916759237875}. Best is trial 0 with value: 1086.156216293503.
[I 2026-02-12 05:51:16,849] Trial 2 finished with value: 1096.4045735551142 and parameters: {'n_estimators': 53, 'learning_rate': 0.12763444772494534, 'max_depth': 6, 'min_samples_leaf': 18, 'subsample': 0.9532724988085814}. Best is trial 0 with value: 1086.156216293503.
[I 2026-02-12 05:51:37,055] Trial 3 finished with value: 1087.62309872448

Best RMSE: 1084.0346554599057
Best Params: {'n_estimators': 156, 'learning_rate': 0.028657407835970948, 'max_depth': 4, 'min_samples_leaf': 13, 'subsample': 0.9359333186050794}


In [13]:
best_model = GradientBoostingRegressor(
    **study.best_params,
    random_state=RANDOM_STATE
)

best_model.fit(X_encoded, y)

predictions = best_model.predict(test_encoded)
predictions = np.clip(predictions, 0, None)

submission = pd.DataFrame({
    "Item_Identifier": test_ids["Item_Identifier"],
    "Outlet_Identifier": test_ids["Outlet_Identifier"],
    "Item_Outlet_Sales": predictions
})

submission.to_csv("bigmart_submission_3.csv", index=False)

print("Submission ready.")


Submission ready.


In [14]:
with mlflow.start_run(run_name="Leaderboard_Log"):
    mlflow.log_metric("leaderboard_score", 1157)


In [25]:
!pkill -f mlflow


In [26]:
import subprocess
import os

# Kill any existing mlflow server
!pkill -f mlflow

# Start MLflow UI in background properly
subprocess.Popen(
    ["mlflow", "ui", "--host", "0.0.0.0", "--port", "5000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)


<Popen: returncode: None args: ['mlflow', 'ui', '--host', '0.0.0.0', '--port...>

In [27]:
from google.colab import output
output.serve_kernel_port_as_iframe(5000)

<IPython.core.display.Javascript object>